In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scite.beam.continuous import ContinuousBeamModes

## Reference cross-section and beam

Rectangular 300 × 700 mm, C30/37, B500B.  
Top (hogging) reinforcement $A_{s,top} = 1610\,\text{mm}^2$ (4×Ø22);  
bottom (sagging) $A_{s,bot} = 1257\,\text{mm}^2$ (4×Ø20).  
Equal spans $L = 6\,\text{m}$.  Matches notebook `07_continuous_beam_kappa_hog`.

Three material modes via `ContinuousBeamModes.for_rc`:
- **design** — EC2 parabola-rectangle, $\gamma_c = 1.5$, $\alpha_{cc} = 0.85$, $\gamma_s = 1.15$
- **mean** — EC2 Sargin ($f_{cm} = f_{ck} + 8$), $\gamma_s = 1.0$

In [ ]:
b_ref, h_ref = 300.0, 700.0
A_s_bot_ref  = 1257.0      # 4×Ø20
A_s_top_ref  = 1610.0      # 4×Ø22
z_s_bot      = 50.0
z_s_top      = h_ref - z_s_bot   # 650.0 mm from bottom
L_ref        = 6000.0

triple = ContinuousBeamModes.for_rc(
    b=b_ref, h=h_ref,
    A_s_bot=A_s_bot_ref, z_s_bot=z_s_bot,
    A_s_top=A_s_top_ref, z_s_top=z_s_top,
    f_ck=30.0, f_yk=500.0,
    L_a=L_ref, L_b=L_ref,
)

cba_d = triple.design
cba_m = triple.mean

## Ultimate load and rotation summary

**Rotation decomposition** for the hogging zone:

$$\theta_u = \int_{\kappa < 0} |\kappa(x)|\,dx, \qquad
  \theta_{el} = \theta_u\big|_{\kappa_{hog}=\kappa_{cr}}, \qquad
  \theta_{pl} = \theta_u - \theta_{el}$$

$\theta_{el}$ is evaluated at the cracking load (onset of hogging cracking); $\theta_{pl}$ is the inelastic (post-cracking) part.

In [ ]:
q_d = cba_d.q_hog_u   # N/mm = kN/m
q_m = cba_m.q_hog_u

dc_d = cba_d._design_chars()
dc_m = cba_m._design_chars()

theta_u_d  = cba_d.theta_u_at_step(-1)
theta_el_d = cba_d.theta_el
theta_pl_d = cba_d.theta_pl_at_step(-1)

theta_u_m  = cba_m.theta_u_at_step(-1)
theta_el_m = cba_m.theta_el
theta_pl_m = cba_m.theta_pl_at_step(-1)

kappa_hog_d = float(cba_d.sweep['kappa_hog'][-1]) * 1e3    # ‰/mm
kappa_hog_m = float(cba_m.sweep['kappa_hog'][-1]) * 1e3

def _print_mode(label, cba, q, dc, kappa_hog, theta_u, theta_el, theta_pl):
    print(f'  {label}')
    print(f'    q_hog_u   = {q:.4f} kN/m    M_R,hog = {cba.M_nl_hog_u:.2f} kN·m')
    print(f'    x/d       = {cba.xd_hog_u:.3f}          ω = {dc["omega"]:.3f}')
    print(f'    κ_R,hog   = {kappa_hog:.4f} ‰/mm')
    print(f'    θ_el      = {theta_el*1e3:.3f} mrad')
    print(f'    θ_u       = {theta_u*1e3:.3f} mrad')
    print(f'    θ_pl      = {theta_pl*1e3:.3f} mrad')

print('=== ULS state — hogging section governs ===')
_print_mode('design', cba_d, q_d, dc_d, kappa_hog_d, theta_u_d, theta_el_d, theta_pl_d)
print()
_print_mode('mean  ', cba_m, q_m, dc_m, kappa_hog_m, theta_u_m, theta_el_m, theta_pl_m)
print()
print(f'  ratio q_m / q_d = {q_m / q_d:.3f}')

## Bending moment profile at ULS

In [ ]:
def _ann_M(ax, color, xd, omega):
    ax.text(0.01, 0.03,
        fr'$x/d$ = {xd:.3f}   $\omega$ = {omega:.3f}',
        transform=ax.transAxes, ha='left', va='bottom', fontsize=9, color=color,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.90))

fig, (ax_d, ax_m) = plt.subplots(1, 2, figsize=(14, 4))

cba_d.plot_moment_profile_at_step(ax=ax_d, kappa_idx=-1, color='#d62728')
ax_d.set_title(fr'$M(x)$ — design  ($q_d$ = {q_d:.3f} kN/m)', fontsize=10)
_ann_M(ax_d, '#d62728', cba_d.xd_hog_u, dc_d['omega'])

cba_m.plot_moment_profile_at_step(ax=ax_m, kappa_idx=-1, color='#2ca02c')
ax_m.set_title(fr'$M(x)$ — mean  ($q_m$ = {q_m:.3f} kN/m)', fontsize=10)
_ann_M(ax_m, '#2ca02c', cba_m.xd_hog_u, dc_m['omega'])

plt.tight_layout()
plt.show()

## Curvature profile at ULS

In [ ]:
def _ann_kappa(ax, color, xd, omega, kappa_hog, theta_el, theta_u, theta_pl):
    # bottom-left: section characteristics
    ax.text(0.01, 0.03,
        fr'$x/d$ = {xd:.3f}   $\omega$ = {omega:.3f}',
        transform=ax.transAxes, ha='left', va='bottom', fontsize=8.5, color=color,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.90))
    # upper-right: rotation decomposition
    ax.text(0.99, 0.97,
        fr'$\kappa_{{R,hog}}$ = {kappa_hog:.4f} ‰/mm'
        '\n'
        fr'$\theta_{{el}}$  = {theta_el * 1e3:.3f} mrad'
        '\n'
        fr'$\theta_{{u}}$   = {theta_u  * 1e3:.3f} mrad'
        '\n'
        fr'$\theta_{{pl}}$  = {theta_pl * 1e3:.3f} mrad',
        transform=ax.transAxes, ha='right', va='top', fontsize=8.5, color=color,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.90))

fig, (ax_d, ax_m) = plt.subplots(1, 2, figsize=(14, 4))

cba_d.plot_kappa_profile_at_step(ax=ax_d, kappa_idx=-1, color='#d62728',
                                  show_annotations=False)
ax_d.set_title(fr'$\kappa(x)$ — design  ($q_d$ = {q_d:.3f} kN/m)', fontsize=10)
_ann_kappa(ax_d, '#d62728', cba_d.xd_hog_u, dc_d['omega'],
           kappa_hog_d, theta_el_d, theta_u_d, theta_pl_d)

cba_m.plot_kappa_profile_at_step(ax=ax_m, kappa_idx=-1, color='#2ca02c',
                                  show_annotations=False)
ax_m.set_title(fr'$\kappa(x)$ — mean  ($q_m$ = {q_m:.3f} kN/m)', fontsize=10)
_ann_kappa(ax_m, '#2ca02c', cba_m.xd_hog_u, dc_m['omega'],
           kappa_hog_m, theta_el_m, theta_u_m, theta_pl_m)

plt.tight_layout()
plt.show()